# Solutions · 10 · Renewal processes

Worked solutions to the exercises of [notebook 10](../notebooks/10_renewal_processes.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from engstoch import renewal as rn
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Buses arrive as a renewal process with gamma inter-arrival times of mean 10 min. For shape 0.5, 1, 4 and infinity (deterministic), compute and simulate a passenger's mean wait. What does a schedule with cv 1.5 cost passengers?

In [2]:
rows = []
for shape in (0.5, 1.0, 4.0, 100.0):
    if np.isinf(shape):
        sampler, m2 = (lambda n, r: np.full(n, 10.0)), 100.0
    else:
        sampler, m2 = (lambda n, r, k=shape: r.gamma(k, 10.0 / k, n)), 100.0 * (1 + 1 / shape)
    ins = rn.inspect(sampler, 200.0, 3000, R(int(10 * min(shape, 9))))
    rows.append((shape, 1 / np.sqrt(shape) if np.isfinite(shape) else 0.0, ins["residual"].mean(), rn.mean_residual_life(10.0, m2)))
print(pd.DataFrame(rows, columns=["gamma shape", "cv", "simulated mean wait (min)", "E[X^2]/(2E[X])"]).to_string(index=False, float_format="%.2f"))
print("cv 1.5:", rn.mean_residual_life(10.0, 100 * (1 + 1.5**2)), "min vs 5 min for a perfectly regular service")

 gamma shape   cv  simulated mean wait (min)  E[X^2]/(2E[X])
        0.50 1.41                      15.08           15.00
        1.00 1.00                       9.76           10.00
        4.00 0.50                       6.11            6.25
      100.00 0.10                       5.08            5.05
cv 1.5: 16.25 min vs 5 min for a perfectly regular service


The mean wait is E[X²]/(2E[X]) = (μ/2)(1 + cv²): half a headway for regular buses, a full headway for
Poisson-like ones, and 16 minutes with cv 1.5 although buses still come every 10 minutes on average. Bunching
costs passengers more than a lower frequency would - the argument for headway-based (rather than
timetable-based) bus control.

## Exercise 2

A component has a lognormal lifetime (median 2000 h, sigma 0.6). Planned replacement costs 1, failure 8. Find the optimal replacement age and the saving over run-to-failure.

In [3]:
sigma, median = 0.6, 2000.0
surv = lambda x: stats.lognorm.sf(np.asarray(x), sigma, scale=median)
T = np.linspace(200, 6000, 400)
o = rn.optimal_age_replacement(surv, 1.0, 8.0, T)
mean_life = stats.lognorm.mean(sigma, scale=median)
print(f"optimal replacement age {o['T']:.0f} h, cost rate {o['cost_rate'] * 1000:.4f} per 1000 h; run to failure {8 / mean_life * 1000:.4f} per 1000 h; saving {100 * (1 - o['cost_rate'] * mean_life / 8):.0f} %")
haz = stats.lognorm.pdf(T, sigma, scale=median) / surv(T)
print(f"the lognormal hazard rises then falls: maximum at {T[np.argmax(haz)]:.0f} h")

optimal replacement age 781 h, cost rate 1.8274 per 1000 h; run to failure 3.3411 per 1000 h; saving 45 %
the lognormal hazard rises then falls: maximum at 2976 h


Replacing at about 780 h saves 45 % of the cost of running to failure. The lognormal hazard is not
monotone - it rises to a maximum and then decreases - so very old survivors are actually *better* than middle-
aged ones, and the cost curve flattens beyond the optimum; for a Weibull with shape > 1 the hazard keeps rising.

## Exercise 3

**Implement it yourself:** write the renewal-equation solver from scratch for a given cdf on a uniform grid and check it against the Erlang-2 closed form.

In [4]:
def renewal_eq(cdf, t_end, n):
    h = t_end / n; t = np.arange(n + 1) * h; F = cdf(t); dF = np.diff(F); m = np.zeros(n + 1)
    for k in range(1, n + 1):
        known = sum(dF[j - 1] * 0.5 * ((m[k - j + 1] if j > 1 else 0.0) + m[k - j]) for j in range(1, k + 1))
        m[k] = (F[k] + known) / (1 - 0.5 * dF[0])
    return t, m
F = lambda x: 1 - np.exp(-2 * x) * (1 + 2 * x)
t, m = renewal_eq(F, 5.0, 500)
print("max error vs the Erlang-2 closed form:", np.max(np.abs(m - rn.renewal_function_erlang2(t, 2.0))))
print("max difference from the library:", np.max(np.abs(m - rn.renewal_function(F, 5.0, 500)[1])))

max error vs the Erlang-2 closed form: 1.6665255186953942e-05
max difference from the library: 6.217248937900877e-15


Each m_k depends on all earlier values, so the scheme is a triangular recursion (O(n²) work, which a convolution
by FFT could reduce). The trapezoidal treatment of each cell makes the m_k term appear on both sides; solving
for it gives the division by 1 − dF₁/2.